In [4]:
import pandas as pd
import os
import json
import warnings
import gc
warnings.filterwarnings('ignore')

import json, ast
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

In [5]:
meta_data_path = '/kaggle/input/datasets/b22dckh121lvnth/dataamazon/meta_data_5_core (1).parquet'
review_data_path = '/kaggle/input/datasets/b22dckh121lvnth/dataamazon/review_data_5_core (1).parquet'

meta_df = pd.read_parquet(meta_data_path)
review_df = pd.read_parquet(review_data_path)

print("Meta Data:")
display(meta_df.head())

print("\nReview Data:")
display(review_df.head())

Meta Data:


,title,rating_number,parent_asin,details
0,Team T-Force Vulcan 16GB (2 x 8GB) 288-Pin DDR...,24,B06W2LL3T5,"{""Product Dimensions"": ""5.51 x 0.28 x 1.26 inc..."
1,RQN Bluetooth Music Soft Hat Warm Velvet Beani...,19,B015OIGT02,"{""Package Dimensions"": ""7.99 x 5.43 x 1.46 inc..."
2,Saicoo® USB 3.0 Foldable Superspeed Multi-in-1...,72,B00BW006UQ,"{""Brand"": ""saicoo"", ""Item model number"": ""S020..."
3,"Ruaeoda Printer Cable 100 ft, Long USB Printer...",627,B07791J9QQ,"{""Brand"": ""Ruaeoda"", ""Connector Type"": ""USB Ty..."
4,ASUS ZenBook Flip 13 Ultra Slim Convertible La...,62,B09PSW44JY,"{""Standing screen display size"": ""13.3 Inches""..."



Review Data:


,parent_asin,rating,timestamp,user_id
0,B00000JBAT,5,1999-06-13 22:10:04,AGUH4HQWSFAZEZWWJUPENAITOIAQ
1,B00000J40U,5,1999-07-14 04:51:14,AEUVZ6UTQAPJNFDG3JA2KMGOO32A
2,B00000J4FY,5,1999-07-14 20:53:38,AHCXZZ6R4US43PBVXXVN47OTTVHQ
3,B00000JMO4,5,1999-07-23 18:26:11,AEIOFVIDBZV56KMYE5666IJ6W3RQ
4,B00000J4FY,5,1999-08-24 22:21:16,AH2IRG5CGMQR5LITB63RU5XNZCLA


## Train / Val / Test Split (Leave-One-Out)
Dùng leave-one-out theo timestamp:

* test → interaction mới nhất của mỗi user
* val → interaction thứ hai từ cuối
* train → phần còn lại
* Sau đó lọc val/test để chỉ giữ (user, item) đều đã xuất hiện trong train (tránh cold-start ảnh hưởng kết quả đánh giá).

In [6]:
print("Tạo train/val/test split (leave-one-out theo timestamp)...")

review_sorted = review_df.sort_values(["user_id", "timestamp"])
review_sorted["_rank"] = (
    review_sorted.groupby("user_id").cumcount(ascending=False)
)  # 0 = last, 1 = second-to-last, ...

test_df  = review_sorted[review_sorted["_rank"] == 0].drop(columns="_rank").reset_index(drop=True)
val_df   = review_sorted[review_sorted["_rank"] == 1].drop(columns="_rank").reset_index(drop=True)
train_df = review_sorted[review_sorted["_rank"] >= 2].drop(columns="_rank").reset_index(drop=True)

print(f"Train : {len(train_df):,} ({len(train_df)/len(review_df)*100:.1f}%)")
print(f"Val   : {len(val_df):,}  ({len(val_df)/len(review_df)*100:.1f}%)")
print(f"Test  : {len(test_df):,}  ({len(test_df)/len(review_df)*100:.1f}%)")
gc.collect()

Tạo train/val/test split (leave-one-out theo timestamp)...
Train : 8,260,845 (78.3%)
Val   : 1,145,516  (10.9%)
Test  : 1,145,516  (10.9%)


510

## Lọc Val/Test: Chỉ Giữ Item Có Trong Train

LightGCN chỉ có embedding cho các entity xuất hiện trong tập train. Bất kỳ item nào không có trong train sẽ không được embed → phải loại khỏi val/test trước khi đánh giá.

In [7]:
train_items = set(train_df["parent_asin"])

def filter_split(df: pd.DataFrame, valid_items: set,
                 split_name: str) -> pd.DataFrame:
    df = df[df["parent_asin"].isin(valid_items)].reset_index(drop=True)
    return df


val_df  = filter_split(val_df, train_items, "Val ")
test_df = filter_split(test_df, train_items, "Test")

print(f"   Train users : {train_df['user_id'].nunique():,}")
print(f"   Train items : {train_df['parent_asin'].nunique():,}")

   Train users : 1,145,516
   Train items : 283,184


## Index Mapping cho LightGCN

LightGCN cần **user_idx** và **item_idx** liên tục bắt đầu từ 0.  
Item index được offset bởi `n_users` để xây đồ thị bipartite trên một không gian node chung.

```
 node_id = user_idx          nếu là user  (0 .. n_users-1)
 node_id = n_users + item_idx nếu là item  (n_users .. n_users+n_items-1)
```

In [8]:
all_users = sorted(train_df["user_id"].unique())
all_items = sorted(train_df["parent_asin"].unique())

user2idx = {u: i for i, u in enumerate(all_users)}
item2idx = {it: i for i, it in enumerate(all_items)}

n_users = len(user2idx)
n_items = len(item2idx)

print(f"n_users : {n_users:,}")
print(f"n_items : {n_items:,}")
print(f"n_nodes (bipartite graph) : {n_users + n_items:,}")


def apply_mapping(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["user_idx"] = df["user_id"].map(user2idx)
    df["item_idx"] = df["parent_asin"].map(item2idx)
    df["user_node"] = df["user_idx"]
    df["item_node"] = df["item_idx"] + n_users
    return df


train_df = apply_mapping(train_df)
val_df   = apply_mapping(val_df)
test_df  = apply_mapping(test_df)

print(train_df[["user_id", "user_idx", "user_node", "parent_asin", "item_idx", "item_node", "rating"]].head(5))

n_users : 1,145,516
n_items : 283,184
n_nodes (bipartite graph) : 1,428,700
                        user_id  user_idx  user_node parent_asin  item_idx  \
0  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B0007MGG2M      5181   
1  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B000XT3L7W     12346   
2  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B00297IPUY     20107   
3  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B000JREMTO      8992   
4  AE22236AFRRSMQIKGG7TPTB75QEA         0          0  B005UP82KU     42907   

   item_node  rating  
0    1150697       5  
1    1157862       5  
2    1165623       5  
3    1154508       5  
4    1188423       5  


## Embedding Item

In [9]:
IGNORE_FIELDS = {
    "Date First Available",
    "Item Weight",
    "Product Dimensions",
    "Package Dimensions",
    "Item Dimensions  LxWxH",
    "Item Dimensions LxWxH",
    "Is Discontinued By Manufacturer",
    "Item model number",
    "Unit Count",
    "Other display features",
    "Standing screen display size",
}

In [10]:
def safe_parse(val: str) -> dict:
    if not isinstance(val, str) or not val.strip():
        return {}
    try:
        return json.loads(val)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(val)
        except Exception:
            return {}


def details_to_text(details: dict) -> str:
    parts = []
    for key, val in details.items():
        if key in IGNORE_FIELDS:
            continue
        if key == "Best Sellers Rank" and isinstance(val, dict):
            parts.append(f"Category: {', '.join(val.keys())}")
            continue
        if isinstance(val, dict):
            val = ", ".join(f"{k}: {v}" for k, v in val.items())
        elif isinstance(val, list):
            val = ", ".join(str(v) for v in val)
        val = str(val).strip()
        if val.lower() not in ("", "n/a", "none", "no", "-"):
            parts.append(f"{key}: {val}")
    return ". ".join(parts)


def build_embedding_text(row) -> str:
    parts = []
    title = str(row.get("title", "")).strip()
    if title:
        parts.append(title)
    details_text = details_to_text(row.get("details_parsed", {}))
    if details_text:
        parts.append(details_text)
    return ". ".join(parts)

In [11]:
# Lấy meta của các item trong train, sắp xếp theo item_idx
train_meta_df = (
    meta_df[meta_df["parent_asin"].isin(set(all_items))]
    .copy()
    .assign(item_idx=lambda d: d["parent_asin"].map(item2idx))
    .sort_values("item_idx")
    .reset_index(drop=True)
)

# Kiểm tra đủ n_items và đúng thứ tự
assert len(train_meta_df) == n_items, f"Thiếu meta: {len(train_meta_df)} vs {n_items}"
assert (train_meta_df["item_idx"].values == list(range(n_items))).all(), "item_idx không liên tục"

# Build embedding text
train_meta_df["details_parsed"] = train_meta_df["details"].apply(safe_parse)
train_meta_df["embedding_input"] = train_meta_df.apply(build_embedding_text, axis=1)

failed = (train_meta_df["details_parsed"].apply(lambda x: x == {})).sum()
print(f"Parse OK: {n_items - failed:,} | Thất bại: {failed:,}")
print(f"Avg text length: {train_meta_df['embedding_input'].str.len().mean():.0f} chars")

# Encode
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)

texts      = train_meta_df["embedding_input"].tolist()
total      = len(texts)
batch_size = 1024
n_batch    = (total + batch_size - 1) // batch_size
all_embs   = []

for i in range(n_batch):
    start = i * batch_size
    end   = min(start + batch_size, total)
    embs  = model.encode(
        texts[start:end],
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    all_embs.append(embs)
    if (i + 1) % 10 == 0 or (i + 1) == n_batch:
        print(f"  Batch {i+1}/{n_batch} — {end:,}/{total:,} items")

embeddings = np.vstack(all_embs)

Parse OK: 280,254 | Thất bại: 2,930
Avg text length: 391 chars


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Batch 10/277 — 10,240/283,184 items
  Batch 20/277 — 20,480/283,184 items
  Batch 30/277 — 30,720/283,184 items
  Batch 40/277 — 40,960/283,184 items
  Batch 50/277 — 51,200/283,184 items
  Batch 60/277 — 61,440/283,184 items
  Batch 70/277 — 71,680/283,184 items
  Batch 80/277 — 81,920/283,184 items
  Batch 90/277 — 92,160/283,184 items
  Batch 100/277 — 102,400/283,184 items
  Batch 110/277 — 112,640/283,184 items
  Batch 120/277 — 122,880/283,184 items
  Batch 130/277 — 133,120/283,184 items
  Batch 140/277 — 143,360/283,184 items
  Batch 150/277 — 153,600/283,184 items
  Batch 160/277 — 163,840/283,184 items
  Batch 170/277 — 174,080/283,184 items
  Batch 180/277 — 184,320/283,184 items
  Batch 190/277 — 194,560/283,184 items
  Batch 200/277 — 204,800/283,184 items
  Batch 210/277 — 215,040/283,184 items
  Batch 220/277 — 225,280/283,184 items
  Batch 230/277 — 235,520/283,184 items
  Batch 240/277 — 245,760/283,184 items
  Batch 250/277 — 256,000/283,184 items
  Batch 260/277 — 

## Xuất file

In [12]:
np.save("embeddings.npy", embeddings)

train_df.to_parquet("train.parquet", index=False)
val_df.to_parquet("val.parquet", index=False)
test_df.to_parquet("test.parquet", index=False)

id_mappings = {
    "user2idx": user2idx,
    "item2idx": item2idx,
    "n_users": n_users,
    "n_items": n_items,
}

with open(f"id_mappings.json", "w") as f:
    json.dump(id_mappings, f, ensure_ascii=False)